In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import warnings
import os
warnings.filterwarnings('ignore')

# Quantum ML libraries
import pennylane as qml
from pennylane import numpy as pnp
from pennylane.optimize import NesterovMomentumOptimizer, AdamOptimizer

print(f"PennyLane version: {qml.__version__}")
print(f"NumPy version: {np.__version__}")


PennyLane version: 0.44.1
NumPy version: 2.4.4


In [10]:
# Load data (works on both Kaggle and local)
import os

if os.path.exists('/kaggle/input/titanic/train.csv'):
    # Kaggle environment
    DATA_PATH = '/kaggle/input/titanic'
else:
    # Local environment
    DATA_PATH = './data'

train_df = pd.read_csv(f'{DATA_PATH}/train.csv')
test_df = pd.read_csv(f'{DATA_PATH}/test.csv')

print(f"Training set: {train_df.shape}")
print(f"Test set: {test_df.shape}")
train_df.head()

Training set: (891, 12)
Test set: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [11]:
def preprocess_data(df, is_train=True):
    """
    Preprocess Titanic data for quantum classification.
    
    For quantum circuits, we need to:
    1. Handle missing values
    2. Encode categorical variables
    3. Select features (limited by qubit count)
    4. Normalize features to [0, pi] for angle encoding
    """
    df = df.copy()
    
    # Fill missing values
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
    
    # Feature engineering
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 
                                        'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace('Mlle', 'Miss')
    df['Title'] = df['Title'].replace('Ms', 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')
    
    # Encode categorical
    df['Sex'] = LabelEncoder().fit_transform(df['Sex'])
    df['Embarked'] = LabelEncoder().fit_transform(df['Embarked'])
    df['Title'] = LabelEncoder().fit_transform(df['Title'])
    
    # Select features for quantum model (limited qubits = limited features)
    # We select 4 features for a 4-qubit circuit
    features = ['Pclass', 'Sex', 'Age', 'Fare']
    
    X = df[features].values
    y = df['Survived'].values if is_train else None
    
    return X, y, df

X_full, y_full, train_processed = preprocess_data(train_df, is_train=True)
print(f"Features shape: {X_full.shape}")
print(f"Feature columns: Pclass, Sex, Age, Fare")

Features shape: (891, 4)
Feature columns: Pclass, Sex, Age, Fare


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_full)

def normalize_to_angle(X):
    X_bounded = np.arctan(X) / (np.pi/2)
    X_angle = (X_bounded + 1) * (np.pi / 2) / 2
    return X_angle

X_quantum = normalize_to_angle(X_scaled)

X_train, X_val, y_train, y_val = train_test_split(X_quantum, y_full, test_size=0.2, random_state=42, stratify=y_full)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Feature range: [{X_quantum.min():.3f}, {X_quantum.max():.3f}]")

In [5]:
X_train_classical, X_val_classical, _, _ =train_test_split(X_scaled, y_full, test_size=0.2, random_state=42, stratify=y_full)

classical_models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000),
    'SVM (RBF)' : SVC(kernel='rbf', probability=True),
    'Random Forest' : RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting' : GradientBoostingClassifier(n_estimators=100, random_state=42)
    }

classical_results = {}
print("Classical Model Performance (4 features only)")

NameError: name 'X_scaled' is not defined